# g2p-agent walkthrough

A guided tour of the retrieval-augmented Claude agent over Broad **Genomics 2 Proteins (G2P)** data.

This notebook runs end-to-end **with no API key** using the deterministic offline backends
(`HashingEmbedder` + `MockLLM`). Set `ANTHROPIC_API_KEY` and install the `embeddings` extra to
use Claude + BGE instead — no code changes needed.

> Not for clinical use. Research/exploration aid only.

## 0. Setup

Run from the repo root in the project venv. Confirm which backends resolved:

In [ ]:
from g2p_agent.config import settings
print('embedder ->', settings.resolve_embedder())
print('llm      ->', settings.resolve_llm())
print('chroma   ->', settings.chroma_dir)

## 1. Ingest G2P data

Pull per-residue protein-feature tables from the G2P portal (cached under `data/raw/`),
chunk by UniProt domain + variant cluster, embed, and write to Chroma.

*(Needs network the first time; cached afterwards. Skip if you already ran `g2p-agent ingest`.)*

In [ ]:
from g2p_agent.ingest import ingest
manifest = ingest(['TP53', 'PTEN'], reset=True, progress=print)
manifest

## 2. Retrieve

Hybrid dense (BGE/hash) + sparse (BM25) retrieval with reciprocal rank fusion and a
residue/gene-aware rerank. Note how the chunk covering residue 175 is boosted to the top.

In [ ]:
from g2p_agent.retrieve import Retriever
r = Retriever()
for s in r.search('DNA-binding domain stability buried', gene='TP53', position=175)[:4]:
    print(f"{s.score:.4f}  {s.chunk.id}  covers175={s.chunk.covers(175)}")
    print('   components:', s.components)

## 3. Inspect a chunk

Chunk text is synthesized from real G2P columns — domain, region context, secondary
structure, AlphaFold pLDDT, and per-residue site/PTM/buried highlights.

In [ ]:
hit = r.search('TP53 R175 zinc DNA binding', gene='TP53', position=175)[0]
print(hit.chunk.id)
print(hit.chunk.text)

## 4. Ask the agent

The agent runs a tool-use loop (`search_variants` → `get_variant_context` → compose),
cites every claim by chunk id, and reports a calibrated confidence.

In [ ]:
from g2p_agent.agent import Agent
resp = Agent().ask('What does the missense variant R175H in TP53 do to protein stability?')
print('CONFIDENCE:', resp.confidence.value, '—', resp.confidence_reasoning)
print('ANSWER:', resp.answer[:400])
for c in resp.claims:
    print('  •', c.text[:90], '->', [ct.chunk_id for ct in c.citations])

## 5. Refusal behavior

When no retrieved chunk supports an answer, the agent must decline rather than guess.

In [ ]:
resp = Agent().ask('What does the EGFR T790M variant do to drug binding?')  # EGFR not ingested
print('insufficient_evidence:', resp.insufficient_evidence)
print(resp.answer)

## 6. Evaluate

Score a few benchmark items. Metrics: task success, grounding, hallucination, calibration.
(The full 32-item baseline lives in `eval/results/baseline.md`.)

In [ ]:
from g2p_agent.eval import load_benchmark, run_eval
items = [i for i in load_benchmark('eval/benchmark.jsonl') if i.gene in ('TP53','PTEN')][:6]
report = run_eval(items, progress=print)
report['metrics']

---

**Next:** see `AGENTS.md` to add tools, metrics, or data sources. Cite Kwon et al., *Nat Methods* 2024 (doi:10.1038/s41592-024-02409-0).